# EDA — Encuesta
Lee el dataset transformado desde `staging/` (columnas ya normalizadas).

In [58]:
import pandas as pd
from pathlib import Path

STAGING_DIR = Path('..') / 'staging'

# Toma el archivo staging más reciente (excluye el diccionario)
staging_files = sorted(
    [f for f in STAGING_DIR.glob('*.xlsx') if 'dictionary' not in f.name],
    key=lambda f: f.stat().st_mtime,
    reverse=True
)

if not staging_files:
    raise FileNotFoundError('No hay archivos en staging/. Corre primero pipeline.py')

latest = staging_files[0]
print(f'Leyendo: {latest.name}')

df = pd.read_excel(latest)
print(f'Shape: {df.shape}')
df.head()

Leyendo: survey_staging_fake.xlsx
Shape: (1000, 18)


,marca_temporal,1_género_colocar_desde_la_base_de_datos,2_rango_etario_seleccionar_desde_la_base_de_datos,3_ciudad_escribir_ej_quito,4_vivienda_y_habitabilidad_rural_referido_a_campo_y_periferias_de_ciudades_identificadas_como_tal_urbano_ciudades_y_centros_cantonales_o_provinciales,5_actividad_económica_de_la_persona_afiliada_indicar_la_o_las_principales_que_generan_ingresos_al_hogar,6_ingreso_promedio_mensual_tomando_como_referencia_500_usd,7_no_de_personas_aseguradas_en_el_núcleo_familiar_escribir_número_sin_espacios_ej_4,8_grupos_de_atención_prioritaria_en_el_núcleo_asegurado,9_uso_del_plan_salud_red_en_el_último_año,10_necesidad_del_asegurado_a_en_acudir_a_chequeos_médicos,11_experiencia_de_uso_del_seguro_en_la_atención_de_una_enfermedad_o_emergencia_colocar_las_ideas_centrales_y_palabras_claves_mencionadas_por_la_persona_encuestada_ej_atención_de_un_parto_buena_experiencia_con_el_seguro_cobertura_de_hospitalización_y_gastos_al_100_opinión_regular_sobre_la_clínica_utilizada_dificultad_en_llenar_formularios_de_reembolso_colocar_n_a_en_caso_que_la_persona_encuestada_no_haya_brindado_información_registrar_experiencias_tanto_positivas_como_negativas,12_antes_de_tener_el_plan_de_salud_plan_salud_red_en_caso_de_enfermedad_o_emergencias_la_familia,13_percepción_del_estado_de_salud_del_afiliado_a,14_gasto_en_salud_tomando_en_cuenta_el_uso_del_plan_salud_red,15_promedio_mensual_de_presupuesto_gastado_en_salud_en_el_núcleo_familiar,16_disponibilidad_de_otros_seguros_para_acceso_a_salud,17_percepción_de_respaldo_en_acceso_a_la_salud_debido_a_la_afiliación_al_plan_salud_red_de_salud
0,11/01/2026 16:07:00,Masculino,45 a 54 años,Ciudad Norte,Urbano,Ama de casa / Cuidado del hogar,Menos de 1 salario básico,6,Niños o niñas menores de 5 años,Sí,Acude y esta en tratamiento por alguna enferme...,MUY BUENA EXPERIENCIA CON EL PLAN,No accedía a ningún servicio de salud,Mejoró poco,Gasta menos,Entre $50 y $100 dólares,No dispone de otro seguro,"Sabe que cuenta con el servicio, pero le preoc..."
1,21/01/2026 22:37:00,Masculino,65 o más años,Montaña Alta,Urbano,"Empleado público (del Estado, gobiernos locale...",Al menos 1 salario básico,3,Adultos mayores (más de 65 años),No,Acude y esta en tratamiento por alguna enferme...,COBERTURA COMPLETA EN CIRUGÍA RECIENTE,Intentó recibir (o recibió) atención en centro...,Mejoró mucho,Gasta igual (no percibe un cambio),Entre $50 y $100 dólares,No dispone de otro seguro,Se siente respaldado
2,20/01/2026 03:38:00,Masculino,35 a 44 años,Costa Nueva,Urbano,Trabajo informal / Jornalero,Menos de 1 salario básico,4,No Aplica,Sí,Acude y esta en tratamiento por alguna enferme...,"REGULAR, TUVO DIFICULTADES CON EL PROCESO DE R...",No accedía a ningún servicio de salud,Ha mejorado,Gasta poco,Entre $50 y $100 dólares,No dispone de otro seguro,Se siente respaldado
3,16/01/2026 21:59:00,Femenino,35 a 44 años,Bahía Grande,Rural,"Empleado público (del Estado, gobiernos locale...",Entre 1 y 2 salarios básicos,5,Niños o niñas menores de 5 años,Sí,Acude solo en caso de emergencias o malestar,BUENA EXPERIENCIA CON EL SERVICIO MÉDICO,Utilizaba medicina alternativa o remedios caseros,Empeoró,Gasta igual (no percibe un cambio),Entre $10 y $50 dólares,Seguro Campesino,Se siente respaldado
4,16/01/2026 19:53:00,Femenino,55 a 64 años,Ciudad Norte,Rural,Ama de casa / Cuidado del hogar,Más de 2 salarios básicos,5,Niños o niñas menores de 5 años,Sí,Acude y esta en tratamiento por alguna enferme...,NaN,Pagaba de su bolsillo (ahorros o presupuesto f...,Ha mejorado,Gasta igual (no percibe un cambio),Entre $10 y $50 dólares,IESS,Se siente respaldado


In [59]:
# Strip a TODAS las columnas de texto del dataframe de una vez
str_cols = df.select_dtypes(include='object').columns
df[str_cols] = df[str_cols].apply(lambda col: col.str.strip())

print(f"Strip aplicado a {len(str_cols)} columnas.")

Strip aplicado a 17 columnas.


In [60]:
# Crear ID correlativo tipo ENC-0001
df.insert(0, 'id', ['ENC-' + str(i).zfill(4) for i in range(1, len(df) + 1)])

print(df['id'].head())

0    ENC-0001
1    ENC-0002
2    ENC-0003
3    ENC-0004
4    ENC-0005
Name: id, dtype: object


In [61]:
for col in df.columns:
    uniques = df[col].dropna().unique()
    print(f"\n{'='*60}")
    print(f"  {col}  ({len(uniques)} únicos)")
    print(f"{'='*60}")
    print(uniques)


  id  (1000 únicos)
['ENC-0001' 'ENC-0002' 'ENC-0003' 'ENC-0004' 'ENC-0005' 'ENC-0006'
 'ENC-0007' 'ENC-0008' 'ENC-0009' 'ENC-0010' 'ENC-0011' 'ENC-0012'
 'ENC-0013' 'ENC-0014' 'ENC-0015' 'ENC-0016' 'ENC-0017' 'ENC-0018'
 'ENC-0019' 'ENC-0020' 'ENC-0021' 'ENC-0022' 'ENC-0023' 'ENC-0024'
 'ENC-0025' 'ENC-0026' 'ENC-0027' 'ENC-0028' 'ENC-0029' 'ENC-0030'
 'ENC-0031' 'ENC-0032' 'ENC-0033' 'ENC-0034' 'ENC-0035' 'ENC-0036'
 'ENC-0037' 'ENC-0038' 'ENC-0039' 'ENC-0040' 'ENC-0041' 'ENC-0042'
 'ENC-0043' 'ENC-0044' 'ENC-0045' 'ENC-0046' 'ENC-0047' 'ENC-0048'
 'ENC-0049' 'ENC-0050' 'ENC-0051' 'ENC-0052' 'ENC-0053' 'ENC-0054'
 'ENC-0055' 'ENC-0056' 'ENC-0057' 'ENC-0058' 'ENC-0059' 'ENC-0060'
 'ENC-0061' 'ENC-0062' 'ENC-0063' 'ENC-0064' 'ENC-0065' 'ENC-0066'
 'ENC-0067' 'ENC-0068' 'ENC-0069' 'ENC-0070' 'ENC-0071' 'ENC-0072'
 'ENC-0073' 'ENC-0074' 'ENC-0075' 'ENC-0076' 'ENC-0077' 'ENC-0078'
 'ENC-0079' 'ENC-0080' 'ENC-0081' 'ENC-0082' 'ENC-0083' 'ENC-0084'
 'ENC-0085' 'ENC-0086' 'ENC-0087' 'ENC-00

## Explosión de columnas de opción múltiple → formato long

In [62]:
# Categorías conocidas por columna (membership check, no split simple)
KNOWN_CATEGORIES = {
    '5_actividad_económica_de_la_persona_afiliada_indicar_la_o_las_principales_que_generan_ingresos_al_hogar': [
        'Emprendimiento o Pequeña/Mediana empresa familiar',
        'Ventas y comercio de mercaderías',
        'Empleado público (del Estado, gobiernos locales o empresas públicas)',
        'Ama de casa / Cuidado del hogar',
        'Profesional independiente / Servicios profesionales',
        'Empleado privado',
        'Actividades agrícolas (producción, cultivos, procesamiento de productos del agro)',
        'Actividades acuícolas (camaroneras, piscinas de crianza de peces)',
        'Actividades ganaderas (crianza de pequeñas, medianas y grandes especies, procesamiento de cárnicos)',
        'Transporte (de mercaderías o bienes, buses, taxis, motoservicios, etc.)',
        'Jubilado o pensionista',
    ],
    '8_grupos_de_atención_prioritaria_en_el_núcleo_asegurado': [
        'Personas con discapacidad (de cualquier tipo)',
        'Adultos mayores (más de 65 años)',
        'Niños o niñas menores de 5 años',
        'Personas con enfermedades crónicas o catastróficas',
        'No Aplica',
    ],
}


def explode_multichoice(df: pd.DataFrame, col: str, categories: list) -> pd.DataFrame:
    rows = []
    for _, row in df[['id', col]].iterrows():
        raw = row[col]
        if pd.isna(raw) or str(raw).strip() == '':
            continue
        found = [cat for cat in categories if cat in str(raw)]
        remaining = str(raw)
        for cat in found:
            remaining = remaining.replace(cat, '')
        otros = [t.strip(' ,') for t in remaining.split(',') if t.strip(' ,')]
        for cat in found:
            rows.append({'id': row['id'], 'categoria': cat})
        if otros:
            rows.append({'id': row['id'], 'categoria': 'Otro'})
    return pd.DataFrame(rows)


COL_5 = '5_actividad_económica_de_la_persona_afiliada_indicar_la_o_las_principales_que_generan_ingresos_al_hogar'
COL_8 = '8_grupos_de_atención_prioritaria_en_el_núcleo_asegurado'

df_long_actividad = explode_multichoice(df, COL_5, KNOWN_CATEGORIES[COL_5])
df_long_grupos    = explode_multichoice(df, COL_8, KNOWN_CATEGORIES[COL_8])

# Columna limpia en df original
def clean_original(val, categories):
    if pd.isna(val):
        return val
    found = [cat for cat in categories if cat in str(val)]
    return val if found else 'Otro'

df[COL_5 + '_clean'] = df[COL_5].apply(lambda x: clean_original(x, KNOWN_CATEGORIES[COL_5]))

print("── df_long_actividad ──")
print(df_long_actividad['categoria'].value_counts().to_string())
print("\n── df_long_grupos ──")
print(df_long_grupos['categoria'].value_counts().to_string())

── df_long_actividad ──
categoria
Emprendimiento o Pequeña/Mediana empresa familiar                       295
Ama de casa / Cuidado del hogar                                         240
Otro                                                                    239
Empleado público (del Estado, gobiernos locales o empresas públicas)    156
Empleado privado                                                         98
Ventas y comercio de mercaderías                                         43

── df_long_grupos ──
categoria
No Aplica                           363
Niños o niñas menores de 5 años     290
Adultos mayores (más de 65 años)    225
Otro                                197


## Normalización pregunta 11 — Experiencia de uso del seguro

In [63]:
COL_11 = '11_experiencia_de_uso_del_seguro_en_la_atención_de_una_enfermedad_o_emergencia_colocar_las_ideas_centrales_y_palabras_claves_mencionadas_por_la_persona_encuestada_ej_atención_de_un_parto_buena_experiencia_con_el_seguro_cobertura_de_hospitalización_y_gastos_al_100_opinión_regular_sobre_la_clínica_utilizada_dificultad_en_llenar_formularios_de_reembolso_colocar_n_a_en_caso_que_la_persona_encuestada_no_haya_brindado_información_registrar_experiencias_tanto_positivas_como_negativas'

EXP_MAP = {
    # ── Positiva - Cirugía ──────────────────────────────────────────────────
    'AFILIADO COMENTA QUE TUVO UNA OPERACIÓN Y SU EXPERIENCIA FUE BUENA COMENTA QUE FUERON EFICIENTES.'
        : 'Positiva - Cirugía',
    'COMENTA QUE PARA UNA OPERACIÓN DE HIJA PARA VESÍCULA Y FUE SATISFACTORIA'
        : 'Positiva - Cirugía',
    'recientemente su hija tuvo una operación en la cual el seguro si le cubrio una gran parte de los gastos y la atención fue buena'
        : 'Positiva - Cirugía',

    # ── Positiva - Hospitalización ──────────────────────────────────────────
    'AFILIADO SATISFECHO CON LA ATENCIÓN RECIBIDA EN UNA HOSPITALIZACIÓN PASADA CON SU ESPOSA'
        : 'Positiva - Hospitalización',
    'El año pasado uno de sus hermanos acudio a un hospital particular pero no le atendieron, despues al usar la cobertura de inmedical recibio atencion hospitalaria y solo tuvieron que pagar la mitad de los gastos.'
        : 'Positiva - Hospitalización',
    'Infección Intestinal con hospitalización, fue buena la atención'
        : 'Positiva - Hospitalización',
    'Le gusta la cobertura que recibe ya que el titular está en un tratamiento que es muy efectivo, incluso este fin de semana tiene programado una cobertura hospitalaria, la cual se la entregan rapido y sin esperar.'
        : 'Positiva - Hospitalización',
    'LE GUSTA EL SEGURO PORQUE RECIENTEMENTE ESTUVO INTERNADA EN UNA CLINICA Y LOS GASTOS LO CUBRIÓ EL SEGURO AL IGUAL QUE LA MEDICACION'
        : 'Positiva - Hospitalización',
    'Hospitalización por emergencia por apendicitis; la atención fue buena.'
        : 'Positiva - Hospitalización',
    'Uno de los hijos recientemente tuvo una hospitalizacion y el seguo logró cubrir parte de los gastos médicos'
        : 'Positiva - Hospitalización',
    'El año pasado tuvo una hospitalizacion uno de los nietos del titular y si recibio una buena atencion y cobertura de parte del seguro'
        : 'Positiva - Hospitalización',
    'Estuvo interna por emergencia, comenta que la experiencia fue buena'
        : 'Positiva - Hospitalización',
    'MYU BUENA TENCION EN LA ATENCION DE 3 DIAS DE HOSPITALIZADA'
        : 'Positiva - Hospitalización',
    'BUENA ATENCION EN EL HOSPITAL'
        : 'Positiva - Hospitalización',

    # ── Positiva - Emergencia ───────────────────────────────────────────────
    'SU ESPOSO FUE ATENDIDO EN UNA EMERGENCIA Y QUEDO MUY SATISFECHA'
        : 'Positiva - Emergencia',
    'EN UNA EMERGENCIA QUE TUVO CON SU ESPOSO FUE BIEN ATENDIDA Y ESTA SATISFECHA'
        : 'Positiva - Emergencia',
    'Un sobrino recibió una atención por emergencia sin tener que esperar y le cubrio la totalidad de la atención y tambien los medicamentos'
        : 'Positiva - Emergencia',
    'FUE BIEN ATENDIDO EL PACIENTE EN EMERGENCIA'
        : 'Positiva - Emergencia',
    'BUENA EMERGENCIA'
        : 'Positiva - Emergencia',
    'MUY BUENA ATENCION EN EMERGENCIA'
        : 'Positiva - Emergencia',

    # ── Positiva - Parto / Maternidad ───────────────────────────────────────
    'TUVO UN PARTO Y UNA ATENCIÓN POR EMERGENCIA Y FUE BIEN ATENDIDO'
        : 'Positiva - Parto / Maternidad',
    'TUVO DOS EXPERIENCIAS DE PARTO EN LA PRIMERA SATISFECHO POR LA COBERTURA Y EN LA SEGUNDA DE LA MISMA MANERA'
        : 'Positiva - Parto / Maternidad',
    'Su esposa estuvo embarazada y pudo cubrir los gastos utilizando la cobertura y la atencion de Plan Salud Red'
        : 'Positiva - Parto / Maternidad',

    # ── Positiva - Odontología ──────────────────────────────────────────────
    'ESTA SATISFECHA CON LA ATENCIÓN EN ODONTOLOGÍA YA QUE ES LA ÚNICA QUE A USADO'
        : 'Positiva - Odontología',
    'Solo ha tenido citas en odontología y ha sido bueno el servicio'
        : 'Positiva - Odontología',

    # ── Positiva - Medicina general ─────────────────────────────────────────
    'UNICAMENTE FUE CON FIEBRE EN UNA OCACION Y FUE BIEN ATENDIDA'
        : 'Positiva - Medicina general',
    'Cuando siente algun malestar, utiliza la cobertura del seguro inmedical y si le ayudan con citas médicas'
        : 'Positiva - Medicina general',
    'ACUDIÓ POR UNA FIEBRE Y ESTA SATISFECHA'
        : 'Positiva - Medicina general',
    'recientemente ha tenido citas médicas y si le entregan el servicio de manera rapida y satisfactoria'
        : 'Positiva - Medicina general',

    # ── Positiva - General ──────────────────────────────────────────────────
    'Le gusta que el seguro ademas de las citas medicas tambien le da cobertura para realizar examenes médicos que son ordenados por el doctor'
        : 'Positiva - General',
    'Hasta ahora es muy buena la cobertura, le han dado atencion cuando lo necesitan y le dan cobertura para examenes médicos'
        : 'Positiva - General',

    # ── Negativa - Cobertura insuficiente ───────────────────────────────────
    'COMENTA QUE HABLANDO DE EXÁMENES NO TIENE COBERTURA PARA RX O LABORATORIO'
        : 'Negativa - Cobertura insuficiente',
    'NO CUBRIA EL SEGURO EL PROCEDIMIENTO QUE SE REALIZO'
        : 'Negativa - Cobertura insuficiente',
    'EL SEGURO NO CUBRIO NADA DE LA CIRUGIA DEL ESPOSO DE LA TITULAR'
        : 'Negativa - Cobertura insuficiente',
    'NO CUBRIO EL SEGURO LA OPERACION DE SU HIJO, NO LE REALIZARON EL REMBOLSO.'
        : 'Negativa - Cobertura insuficiente',
    'NO CUBRE EL MEDICAMENTO NI TIENEN TODAS LAS ESPECIALIDADES'
        : 'Negativa - Cobertura insuficiente',
    'NO CUBRE NADA EL SEGURO'
        : 'Negativa - Cobertura insuficiente',

    # ── Negativa - Acceso a especialidades ──────────────────────────────────
    'AFILIADA COMENTA QUE NO ESTA DE ACUERDO CON QUE TENGA QUE TENER DERIVACIÓN PARA ALUNA ESPECIALIDAD YA QUE LE TOMA TIEMPO RECIBIR LA SIGUIENTE CITA MEDICA.'
        : 'Negativa - Acceso a especialidades',
    'NO ATIENDEN ESPECIALISTAS, EL MISMO MEDICO ATIENDE A TODOS'
        : 'Negativa - Acceso a especialidades',
    'NO TIENE BUENA ATENCIÓN, YA QUE HASTA POR QUERER ODONTOLOGÍA DEBE PASAR POR MG'
        : 'Negativa - Acceso a especialidades',

    # ── Negativa - Atención deficiente ──────────────────────────────────────
    'AFILIADA COMENTA QUE A SOLICITADO CITAS MEDICAS Y NO LE HAN AGENDADO POR FALTA DE PAGO CON LOS CENTROS MÉDICOS QUE ESTÁN CERCANOS, USA POCO EL SEGURO DE INMEDICAL.'
        : 'Negativa - Atención deficiente',
    'ATENCIÓN EN MEDICINA GENERAL SOCIO CONSIDERA QUE FUE POCO SATISFACTORIA ADEMAS QUE LA MEDICINA FUE POCO CUBIERTA PARA LO QUE LE ENVIARON Y NECESITA'
        : 'Negativa - Atención deficiente',
    'OBTUVO UNA MALA EXPERIENCIA AL ACUDIR POR SU ESPOSO PERO NO FUE INTERNADO SITIO QUE NO FUE BIEN ATENDIDA TUVO QUE SOLICITAR QUE LE HICIERAN DEVOLUCIÓN  ECONÓMICA YA QUE LE TOCO SER ATENDIDO E OTRO LUGAR. A PARTE DE ESO SATISFECHA CON LA ATENCIÓN'
        : 'Negativa - Atención deficiente',
    'EN EMERGENCIA LA ATENCION NO FUE BUENA'
        : 'Negativa - Atención deficiente',

    # ── Mixta ────────────────────────────────────────────────────────────────
    'COMENTA QUE ACUDIÓ POR EMERGENCIA CON SIGNOS GRIPALES Y ESTUVO SATISFECHO CON LA COBERTURA DE LA MEDICINA Y LA ATENCIÓN.( COMENTA QUE EN LA FAMILIA TIENEN UNA ENFERMEDAD DE DIABETES TIPO 2 CON LA QUE NO CUENTAN CON LA COBERTURA EN MEDICAMENTOS Y DEMÁS COSAS QUE NECESITAN, POCO SATISFECHOS POR ESE LADO.'
        : 'Mixta',
    'NO TENIA COBERTURA PARA UNA RUPTURA DE TIBIA QUE SUFRIÓ SU HIJO, DE AHÍ COMENTA QUE LA ATENCIÓN ES BUENA'
        : 'Mixta',
    'AFILIADA DIO A LUZ CONTANDO CON EL SEGURO, COMENTA QUE FUE UNA EMERGENCIA QUE NO ESTUVO CUBIERTA EN SU TOTALIDAD Y TUVO QUE CUBRIR LA AFILIADA SIENDO DESPUÉS DEVUELTO POR EL SEGURO'
        : 'Mixta',
    'Le gusta la cobertura que brinda el seguro pero en especialidades no le gusta tener que esperar varios días para recibir una atencion en alguna especialidad'
        : 'Mixta',
    'SU ESPOSA FUE OPERADA Y ESTA SATISFECHO CON LA COBERTURA Y ATENCIÓN.. PERO TAMBIÉN EN CLÍNICA ROMERO COMENTA QUE NO LE GUSTO LA ATENCIÓN.'
        : 'Mixta',
    'BUENA EXPERIENCIA EN MG, MALA EXPERIENCIA EN ODONTOLOGIA Y MAL TRATO DEL MEDICO'
        : 'Mixta',
    'LA COBERTURA DEBERIA SER MAS AMPLIA EN MEDICAMENTOS, LA ATENCION ES BUENA'
        : 'Mixta',

    # ── Sin novedad ──────────────────────────────────────────────────────────
    'SIN NOVEDADES EN LA ATENCION' : 'Sin novedad',
    'SIN NOVEDADES'                : 'Sin novedad',
    'SIN NOVEDAD, TODO MUY BIEN'   : 'Sin novedad',
    'SIN NOVEDAD, BUENA ATENCION'  : 'Sin novedad',
}

POSITIVA_GENERAL_KEYWORDS = [
    'BUENA', 'BUEN', 'MUY BIEN', 'EXCELENTE', 'SATISFECH', 'BIEN ATEND',
    'NINGUNA EN PARTICULAR', 'NO TIENE QUEJAS', 'BUENAS EXPERIENCIAS',
    'BUENA INFORMACION', 'RECIBIO BUENOS', 'TELEMEDICA',
]

def categorize_exp(val):
    if pd.isna(val) or str(val).strip().upper() in ('N/A', 'NA', ''):
        return 'Sin novedad'
    v = str(val).strip()
    if v in EXP_MAP:
        return EXP_MAP[v]
    v_up = v.upper()
    if any(kw in v_up for kw in POSITIVA_GENERAL_KEYWORDS):
        return 'Positiva - General'
    return 'Sin clasificar'

df['11_experiencia_categoria'] = df[COL_11].apply(categorize_exp)

print(df['11_experiencia_categoria'].value_counts().to_string())
print(f"\nSin clasificar:")
print(df[df['11_experiencia_categoria'] == 'Sin clasificar'][COL_11].values)

11_experiencia_categoria
Positiva - General    473
Sin clasificar        325
Sin novedad           202

Sin clasificar:
['COBERTURA COMPLETA EN CIRUGÍA RECIENTE'
 'REGULAR, TUVO DIFICULTADES CON EL PROCESO DE REEMBOLSO'
 'COBERTURA COMPLETA EN CIRUGÍA RECIENTE'
 'AFILIADO CONFORME CON EL PLAN DE SALUD'
 'COBERTURA COMPLETA EN CIRUGÍA RECIENTE'
 'AFILIADO CONFORME CON EL PLAN DE SALUD'
 'AFILIADO CONFORME CON EL PLAN DE SALUD'
 'CONFORME CON EL SERVICIO PERO ESPERA MEJORAS EN TIEMPOS DE ESPERA'
 'CONFORME CON EL SERVICIO PERO ESPERA MEJORAS EN TIEMPOS DE ESPERA'
 'CONFORME CON EL SERVICIO PERO ESPERA MEJORAS EN TIEMPOS DE ESPERA'
 'REGULAR, TUVO DIFICULTADES CON EL PROCESO DE REEMBOLSO'
 'COBERTURA COMPLETA EN CIRUGÍA RECIENTE'
 'AFILIADO CONFORME CON EL PLAN DE SALUD'
 'CONFORME CON EL SERVICIO PERO ESPERA MEJORAS EN TIEMPOS DE ESPERA'
 'ATENCIÓN RÁPIDA Y EFICIENTE EN LA CLÍNICA'
 'REGULAR, TUVO DIFICULTADES CON EL PROCESO DE REEMBOLSO'
 'ATENCIÓN RÁPIDA Y EFICIENTE EN LA CLÍNICA'
 'AT

In [64]:
print(f"\nSin clasificar:")                                                                
print(df[df['11_experiencia_categoria'] == 'Sin clasificar'][COL_11].values)   


Sin clasificar:
['COBERTURA COMPLETA EN CIRUGÍA RECIENTE'
 'REGULAR, TUVO DIFICULTADES CON EL PROCESO DE REEMBOLSO'
 'COBERTURA COMPLETA EN CIRUGÍA RECIENTE'
 'AFILIADO CONFORME CON EL PLAN DE SALUD'
 'COBERTURA COMPLETA EN CIRUGÍA RECIENTE'
 'AFILIADO CONFORME CON EL PLAN DE SALUD'
 'AFILIADO CONFORME CON EL PLAN DE SALUD'
 'CONFORME CON EL SERVICIO PERO ESPERA MEJORAS EN TIEMPOS DE ESPERA'
 'CONFORME CON EL SERVICIO PERO ESPERA MEJORAS EN TIEMPOS DE ESPERA'
 'CONFORME CON EL SERVICIO PERO ESPERA MEJORAS EN TIEMPOS DE ESPERA'
 'REGULAR, TUVO DIFICULTADES CON EL PROCESO DE REEMBOLSO'
 'COBERTURA COMPLETA EN CIRUGÍA RECIENTE'
 'AFILIADO CONFORME CON EL PLAN DE SALUD'
 'CONFORME CON EL SERVICIO PERO ESPERA MEJORAS EN TIEMPOS DE ESPERA'
 'ATENCIÓN RÁPIDA Y EFICIENTE EN LA CLÍNICA'
 'REGULAR, TUVO DIFICULTADES CON EL PROCESO DE REEMBOLSO'
 'ATENCIÓN RÁPIDA Y EFICIENTE EN LA CLÍNICA'
 'ATENCIÓN RÁPIDA Y EFICIENTE EN LA CLÍNICA'
 'CONFORME CON EL SERVICIO PERO ESPERA MEJORAS EN TIEMPOS DE E

In [65]:
df.columns.to_list()

['id',
 'marca_temporal',
 '1_género_colocar_desde_la_base_de_datos',
 '2_rango_etario_seleccionar_desde_la_base_de_datos',
 '3_ciudad_escribir_ej_quito',
 '4_vivienda_y_habitabilidad_rural_referido_a_campo_y_periferias_de_ciudades_identificadas_como_tal_urbano_ciudades_y_centros_cantonales_o_provinciales',
 '5_actividad_económica_de_la_persona_afiliada_indicar_la_o_las_principales_que_generan_ingresos_al_hogar',
 '6_ingreso_promedio_mensual_tomando_como_referencia_500_usd',
 '7_no_de_personas_aseguradas_en_el_núcleo_familiar_escribir_número_sin_espacios_ej_4',
 '8_grupos_de_atención_prioritaria_en_el_núcleo_asegurado',
 '9_uso_del_plan_salud_red_en_el_último_año',
 '10_necesidad_del_asegurado_a_en_acudir_a_chequeos_médicos',
 '11_experiencia_de_uso_del_seguro_en_la_atención_de_una_enfermedad_o_emergencia_colocar_las_ideas_centrales_y_palabras_claves_mencionadas_por_la_persona_encuestada_ej_atención_de_un_parto_buena_experiencia_con_el_seguro_cobertura_de_hospitalización_y_gastos_al_10

## Normalización de zonas geográficas

In [66]:
COL_ZONA = '3_ciudad_escribir_ej_quito'

# Paso 1: title case base
df[COL_ZONA] = df[COL_ZONA].str.strip().str.title()

# Paso 2: diccionario de correcciones (typos y variantes → valor canónico)
corrections = {
    'La Libertdad'                      : 'La Libertad',
    'La Ibertad'                        : 'La Libertad',
    'Santo Domingo De Los Colorados'    : 'Santo Domingo de los Colorados',
    'Santo Domingo De Los Tsachilas'    : 'Santo Domingo de los Colorados',
    'Tulcan'    : 'Tulcán',
    'Buena Fe'    : 'Buena Fé',
    'Duran'    : 'Durán',

}

df[COL_ZONA] = df[COL_ZONA].replace(corrections)

# Resultado
print(df[COL_ZONA].value_counts().to_string())


3_ciudad_escribir_ej_quito
Ciudad Norte      290
Puerto Central    169
Valle Sur         146
Montaña Alta      131
Bahía Grande      101
Costa Nueva        68
Río Verde          68
Laguna Alta        27


In [67]:
MULTI_COLS = [
    '5_actividad_económica_de_la_persona_afiliada_indicar_la_o_las_principales_que_generan_ingresos_al_hogar',
    '8_grupos_de_atención_prioritaria_en_el_núcleo_asegurado',
]

for col in MULTI_COLS:
    print(f"\n{'='*70}")
    print(f"  {col}")
    print(f"{'='*70}")
    for val in df[col].dropna().unique():
        print(repr(val))


  5_actividad_económica_de_la_persona_afiliada_indicar_la_o_las_principales_que_generan_ingresos_al_hogar
'Ama de casa / Cuidado del hogar'
'Empleado público (del Estado, gobiernos locales o empresas públicas)'
'Trabajo informal / Jornalero'
'Emprendimiento o Pequeña/Mediana empresa familiar'
'Agricultura / Pesca / Ganadería'
'Empleado privado'
'Emprendimiento o Pequeña/Mediana empresa familiar, Empleado privado'
'Ventas y comercio de mercaderías, Emprendimiento o Pequeña/Mediana empresa familiar'
'Trabajo informal / Jornalero, Agricultura / Pesca / Ganadería'

  8_grupos_de_atención_prioritaria_en_el_núcleo_asegurado
'Niños o niñas menores de 5 años'
'Adultos mayores (más de 65 años)'
'No Aplica'
'Niños o niñas menores de 5 años, Adultos mayores (más de 65 años)'
'Mujeres embarazadas'
'Mujeres embarazadas, Niños o niñas menores de 5 años'
'Personas con discapacidad'
'Adultos mayores (más de 65 años), Personas con discapacidad'


In [68]:
COL_SN = '9_uso_del_plan_salud_red_en_el_último_año'

# Paso 1: title case base
df[COL_SN] = df[COL_SN].str.strip().str.title()

# Paso 2: diccionario de correcciones (typos y variantes → valor canónico)
corrections = {
    'SI'                      : 'Sí',
    'No'                        : 'No',
}

df[COL_SN] = df[COL_SN].replace(corrections)

# Resultado
print(df[COL_SN].value_counts().to_string())


9_uso_del_plan_salud_red_en_el_último_año
Sí    726
No    274


In [69]:
df_le = df.copy()

In [70]:
col = '7_no_de_personas_aseguradas_en_el_núcleo_familiar_escribir_número_sin_espacios_ej_4'

df_le[col] = df_le[col].clip(upper=6)

In [71]:
df_le[col].unique()  # debe devolver solo [1, 2, 3, 4, 5, 6]

array([6, 3, 4, 5, 1, 2])

In [72]:
# ENC-0311: tenía "Adultos mayores (más de 65 años), No Aplica" en Q8
# → limpiar la columna raw en df_le Y eliminar la fila "No Aplica" del long
COL_8 = '8_grupos_de_atención_prioritaria_en_el_núcleo_asegurado'

df_le.loc[df_le['id'] == 'ENC-0311', COL_8] = 'Adultos mayores (más de 65 años)'

df_long_grupos = df_long_grupos[
    ~((df_long_grupos['id'] == 'ENC-0311') & (df_long_grupos['categoria'] == 'No Aplica'))
].reset_index(drop=True)

print("df_le Q8 para ENC-0311:", df_le.loc[df_le['id'] == 'ENC-0311', COL_8].values)
print(df_long_grupos[df_long_grupos['id'] == 'ENC-0311'])

df_le Q8 para ENC-0311: ['Adultos mayores (más de 65 años)']
Empty DataFrame
Columns: [id, categoria]
Index: []


In [73]:
df_le['6_ingreso_promedio_mensual_tomando_como_referencia_500_usd'].unique()

array(['Menos de 1 salario básico', 'Al menos 1 salario básico',
       'Entre 1 y 2 salarios básicos', 'Más de 2 salarios básicos',
       'Prefiere no responder'], dtype=object)

In [74]:
col = '6_ingreso_promedio_mensual_tomando_como_referencia_500_usd'

mapa_ingreso = {
    'Menos de 1 salario básico':      '1. Menos de 1 salario básico',
    'Al menos 1 salario básico':      '2. Al menos 1 salario básico',
    'Entre 1 y 2 salarios básicos':   '3. Entre 1 y 2 salarios básicos',
    'Más de 2 salarios básicos':      '4. Más de 2 salarios básicos',
    'Prefiere no responder':          '5. Prefiere no responder',
}

df_le[col] = df_le[col].map(mapa_ingreso)

In [75]:
df_le['12_antes_de_tener_el_plan_de_salud_plan_salud_red_en_caso_de_enfermedad_o_emergencias_la_familia'].value_counts()

12_antes_de_tener_el_plan_de_salud_plan_salud_red_en_caso_de_enfermedad_o_emergencias_la_familia
Pagaba de su bolsillo (ahorros o presupuesto familiar)           396
Intentó recibir (o recibió) atención en centro médico público    370
No accedía a ningún servicio de salud                            154
Utilizaba medicina alternativa o remedios caseros                 80
Name: count, dtype: int64

In [76]:
col = '12_antes_de_tener_el_plan_de_salud_plan_salud_red_en_caso_de_enfermedad_o_emergencias_la_familia'

mapa_12 = {
    'IESS':                        'Intentó recibir (o recibió) atención en centro médico público',
    'SEGURO CAMPESINO':            'Intentó recibir (o recibió) atención en centro médico público',
    'USABA SEGURO ISSFA':          'Intentó recibir (o recibió) atención en centro médico público',
    'ANTES TENIAN SEGURO PRIVADO': 'Intentó recibir (o recibió) atención en centro médico público',
    'CLINICAS':                    'Pagaba de su bolsillo (ahorros o presupuesto familiar)',
    'PARTICULAR':                  'Pagaba de su bolsillo (ahorros o presupuesto familiar) ',
    'SE AUTOMEDICABA':             'Pagaba de su bolsillo (ahorros o presupuesto familiar) ',
    'NO MENCIONA':                 'Sin información',
}

df_le[col] = df_le[col].map(mapa_12).fillna(df_le[col])

In [77]:
df_le['15_promedio_mensual_de_presupuesto_gastado_en_salud_en_el_núcleo_familiar'].unique()

array(['Entre $50 y $100 dólares', 'Entre $10 y $50 dólares',
       'Entre $100 y $200 dólares', 'Más de $200 dólares'], dtype=object)

In [78]:
col = '15_promedio_mensual_de_presupuesto_gastado_en_salud_en_el_núcleo_familiar'

mapa_15 = {
    'Entre $10 y $50 dólares':   '1. Entre $10 y $50',
    'Entre $50 y $100 dólares':  '2. Entre $50 y $100',
    'Entre $100 y $200 dólares': '3. Entre $100 y $200',
    'Más de $200 dólares':       '4. Más de $200',
}

df_le[col] = df_le[col].map(mapa_15)

In [79]:
df_le['9_uso_del_plan_salud_red_en_el_último_año'] = df_le['9_uso_del_plan_salud_red_en_el_último_año'].fillna('Si')

In [80]:
df_le['7_no_de_personas_aseguradas_en_el_núcleo_familiar_escribir_número_sin_espacios_ej_4'].unique()


array([6, 3, 4, 5, 1, 2])

In [81]:
df_le['9_uso_del_plan_salud_red_en_el_último_año'].value_counts()

9_uso_del_plan_salud_red_en_el_último_año
Sí    726
No    274
Name: count, dtype: int64

In [82]:
  # Diagnóstico de discrepancia "No aplica"                                                                                              
  cat = 'No Aplica'                                                                                                                                                                                                                                                               
  n_noaplica = (df_long_grupos['categoria'] == cat).sum()                                                                                
  total = len(df_le)

  print(f"n 'No Aplica' en df_long_grupos : {n_noaplica}")
  print(f"Total encuestados (len df_le)   : {total}")
  print(f"% crosstab                      : {n_noaplica/total*100:.4f}%")
  print()
  print(f"IDs con 'No Aplica':")
  print(df_long_grupos[df_long_grupos['categoria'] == cat]['id'].values)

n 'No Aplica' en df_long_grupos : 362
Total encuestados (len df_le)   : 1000
% crosstab                      : 36.2000%

IDs con 'No Aplica':
['ENC-0003' 'ENC-0013' 'ENC-0019' 'ENC-0020' 'ENC-0021' 'ENC-0022'
 'ENC-0024' 'ENC-0025' 'ENC-0026' 'ENC-0030' 'ENC-0031' 'ENC-0032'
 'ENC-0034' 'ENC-0035' 'ENC-0039' 'ENC-0040' 'ENC-0046' 'ENC-0048'
 'ENC-0049' 'ENC-0050' 'ENC-0053' 'ENC-0054' 'ENC-0057' 'ENC-0058'
 'ENC-0062' 'ENC-0063' 'ENC-0065' 'ENC-0067' 'ENC-0073' 'ENC-0074'
 'ENC-0075' 'ENC-0078' 'ENC-0079' 'ENC-0080' 'ENC-0081' 'ENC-0083'
 'ENC-0086' 'ENC-0087' 'ENC-0091' 'ENC-0097' 'ENC-0108' 'ENC-0109'
 'ENC-0117' 'ENC-0121' 'ENC-0123' 'ENC-0124' 'ENC-0126' 'ENC-0129'
 'ENC-0135' 'ENC-0136' 'ENC-0140' 'ENC-0141' 'ENC-0143' 'ENC-0146'
 'ENC-0147' 'ENC-0149' 'ENC-0152' 'ENC-0154' 'ENC-0155' 'ENC-0159'
 'ENC-0165' 'ENC-0168' 'ENC-0172' 'ENC-0177' 'ENC-0185' 'ENC-0189'
 'ENC-0190' 'ENC-0191' 'ENC-0194' 'ENC-0195' 'ENC-0197' 'ENC-0199'
 'ENC-0201' 'ENC-0207' 'ENC-0209' 'ENC-0214' 'ENC-0215

In [83]:
col_p10 = '10_necesidad_del_asegurado_a_en_acudir_a_chequeos_médicos'                                                                                                     
print(df_le[col_p10].value_counts(dropna=False))                                                                                       
print()                                                        
print("Nulos:", df_le[col_p10].isna().sum())                                                                                             
print("IDs con nulo:", df_le[df_le[col_p10].isna()]['id'].values)   

10_necesidad_del_asegurado_a_en_acudir_a_chequeos_médicos
Acude solo en caso de emergencias o malestar                    362
Acude y esta en tratamiento por alguna enfermedad o malestar    292
Acude regularmente                                              198
Considera que se encuentra saludable                            148
Name: count, dtype: int64

Nulos: 0
IDs con nulo: []


In [84]:
print(df_le[df_le[col_p10].isna()]['id'].values) 

[]


In [85]:
import sys
sys.path.append('..')

from src.upload_to_sheets import upload_all

upload_all(df_le)

Subiendo 'data' (1000 filas × 21 cols)... ✓
Subiendo 'diccionario' (18 columnas)... ✓

Listo → https://docs.google.com/spreadsheets/d/1j94L-TVr5K-ypWATTyKpML6ip0TwtAFBKFMeTq7gWmI


## Tablas de cruce (crosstabs)

In [86]:
import importlib
import src.crosstabs as _ct
importlib.reload(_ct)
from src.crosstabs import generate_all

generate_all(df_le, df_long_actividad, df_long_grupos)

── Frecuencias univariadas ──
  genero                     Género (P1)
  rango_etario               Rango etario (P2)
  ciudad                     Ciudad (P3)
  vivienda                   Vivienda y habitabilidad (P4)
  ingreso                    Ingreso promedio mensual (P6)
  n_asegurados               Nº personas aseguradas (P7)
  necesidad                  Necesidad de chequeos médicos (P10)
  experiencia                Experiencia de uso del seguro (P11)
  percepcion_salud           Percepción de salud del afiliado (P13)
  gasto_salud                Gasto en salud (P14)
  promedio_gasto             Promedio mensual gasto en salud (P15)
  otros_seguros              Disponibilidad de otros seguros (P16)
  percepcion_respaldo        Percepción de respaldo (P17)
  actividad                  Actividad económica (P5)
  grupos                     Grupos atención prioritaria (P8)

── Tablas de cruce ──
  Q2xQ1         Rango etario (P2)  ×  Género (P1)
  Q4xQ2         Vivienda y habitabili

'C:\\Users\\aizag\\proyectos\\survey-analysis-pipeline\\output\\crosstabs.xlsx'